# EPIC-KITCHENS-100: Data Exploration and Description

This notebook provides a comprehensive exploration of the EPIC-KITCHENS-100 dataset used for the Multi-Instance Retrieval challenge.  
It covers:
- Dataset sources and formats
- Annotation statistics and distributions
- Image/frame statistics (pixel averages, sizes, resolutions)
- Video statistics (duration, resolution, FPS)
- Visualisations and sample images

> **Note**: This notebook works with *both* real data (when available in `data/raw/EK100/`) and a
> synthetic placeholder dataset for CI/testing purposes. Sections that require real data are clearly
> marked and will be skipped gracefully when the data is not present.

## How to get the data
```bash
# Annotations only (~50 MB)
python data/download/download_annotations.py

# Full media (RGB frames recommended; ~740 GB full, use --participants for a subset)
python data/download/download_ek100.py --participants P01 P02 P03 --rgb-frames-only
```

In [ ]:
from __future__ import annotations

import sys
import os
from pathlib import Path

# Ensure the src/ package is importable when running from the notebooks/ directory
REPO_ROOT = Path("__file__").resolve().parent.parent
SRC_PATH = REPO_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

DATA_ROOT = REPO_ROOT / "data" / "raw" / "EK100"
ANN_DIR   = DATA_ROOT / "epic-kitchens-100-annotations"
FRAMES_DIR = DATA_ROOT / "rgb_frames"
VIDEOS_DIR = DATA_ROOT / "videos"

HAS_ANNOTATIONS = ANN_DIR.exists()
HAS_FRAMES      = FRAMES_DIR.exists()
HAS_VIDEOS      = VIDEOS_DIR.exists()

print(f"Annotations available : {HAS_ANNOTATIONS}")
print(f"RGB frames available  : {HAS_FRAMES}")
print(f"Videos available      : {HAS_VIDEOS}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image
import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120

---
## 1. Fuentes de datos disponibles

| Fuente | Descripción | Tamaño aprox. |
|--------|-------------|---------------|
| RGB Frames | Fotogramas extraídos a 256px lado corto, JPEG | ~200 GB |
| Optical Flow | Fotogramas de flujo óptico (u, v), JPEG | ~200 GB |
| Videos originales | Archivos .MP4 en resolución original | ~340 GB |
| Anotaciones | CSVs con metadatos de segmentos de acción | ~50 MB |
| Relevancy matrices | Matrices de correlación semántica (pickle) | <1 MB |

### Estructura de directorios esperada
```
data/raw/EK100/
├── videos/                              # .MP4 por participante
│   └── P{NN}/
│       └── P{NN}_{VV}.MP4
├── rgb_frames/                          # Fotogramas JPEG
│   └── P{NN}/
│       └── P{NN}_{VV}/
│           └── frame_{FFFFFFFFFF}.jpg
└── epic-kitchens-100-annotations/
    ├── EPIC_100_retrieval_train.csv
    ├── EPIC_100_retrieval_test.csv
    ├── EPIC_100_retrieval_train_sentence.csv
    ├── EPIC_100_retrieval_test_sentence.csv
    └── retrieval_annotations/relevancy/
        ├── caption_relevancy_EPIC_100_retrieval_train.pkl
        └── caption_relevancy_EPIC_100_retrieval_test.pkl
```

---
## 2. Estadísticas de anotaciones

In [ ]:
if HAS_ANNOTATIONS:
    train_df = pd.read_csv(ANN_DIR / "EPIC_100_retrieval_train.csv")
    test_df  = pd.read_csv(ANN_DIR / "EPIC_100_retrieval_test.csv")
    train_sent_df = pd.read_csv(ANN_DIR / "EPIC_100_retrieval_train_sentence.csv")
    test_sent_df  = pd.read_csv(ANN_DIR / "EPIC_100_retrieval_test_sentence.csv")

    print("=== Train split ===")
    print(f"  Segments : {len(train_df):,}")
    print(f"  Captions : {len(train_sent_df):,}")
    print(f"  Columns  : {list(train_df.columns)}")
    print()
    print("=== Test split ===")
    print(f"  Segments : {len(test_df):,}")
    print(f"  Captions : {len(test_sent_df):,}")
    display(train_df.head(5))
else:
    # Synthetic placeholder — mirrors the real CSV schema
    np.random.seed(42)
    N = 500
    participants = [f"P{i:02d}" for i in range(1, 38)]
    videos = [f"{p}_{j:03d}" for p in participants for j in range(1, 4)]
    np.random.shuffle(videos)

    train_df = pd.DataFrame({
        "narration_id"   : [f"n{i:06d}" for i in range(N)],
        "participant_id" : np.random.choice(participants, N),
        "video_id"       : np.random.choice(videos, N),
        "start_timestamp": pd.to_timedelta(np.random.uniform(0, 1800, N), unit="s").astype(str),
        "stop_timestamp" : pd.to_timedelta(np.random.uniform(0, 1800, N), unit="s").astype(str),
        "narration"      : [f"Sample action description {i}" for i in range(N)],
        "verb"           : np.random.choice(["take","put","open","close","wash"], N),
        "noun"           : np.random.choice(["pan","cup","knife","plate","fridge"], N),
    })
    test_df = train_df.sample(100).reset_index(drop=True)
    train_sent_df = pd.DataFrame({"narration_id": train_df["narration_id"], "sentence": train_df["narration"]})
    test_sent_df  = pd.DataFrame({"narration_id": test_df["narration_id"],  "sentence": test_df["narration"]})

    print("[SYNTHETIC DATA — real dataset not found]")
    print(f"Train segments: {len(train_df):,}  |  Test segments: {len(test_df):,}")
    display(train_df.head(5))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Clips per participant
part_counts = train_df["participant_id"].value_counts().head(15)
axes[0].bar(part_counts.index, part_counts.values, color="steelblue")
axes[0].set_title("Clips per Participant (top 15)")
axes[0].set_xlabel("Participant")
axes[0].set_ylabel("Number of clips")
axes[0].tick_params(axis="x", rotation=45)

# Verb distribution
if "verb" in train_df.columns:
    verb_counts = train_df["verb"].value_counts().head(15)
    axes[1].barh(verb_counts.index[::-1], verb_counts.values[::-1], color="coral")
    axes[1].set_title("Top-15 Verbs")
    axes[1].set_xlabel("Count")

# Noun distribution
if "noun" in train_df.columns:
    noun_counts = train_df["noun"].value_counts().head(15)
    axes[2].barh(noun_counts.index[::-1], noun_counts.values[::-1], color="mediumseagreen")
    axes[2].set_title("Top-15 Nouns")
    axes[2].set_xlabel("Count")

plt.tight_layout()
plt.savefig("annotation_stats.png", bbox_inches="tight")
plt.show()

---
## 3. Estadísticas de imágenes (fotogramas RGB)

Analizamos resolución, tamaño y distribución de píxeles.

In [ ]:
import torch
from torchvision import transforms

def _load_frame_pil(path: Path) -> Image.Image:
    return Image.open(path).convert("RGB")

if HAS_FRAMES:
    all_frames = sorted(FRAMES_DIR.glob("**/*.jpg"))[:2000]  # cap at 2000
    source = "real"
else:
    # Generate synthetic frames for illustration
    import io
    import random
    random.seed(42)
    N_SYNTH = 200
    SYNTH_SIZES = [(456, 256), (456, 256), (640, 360), (640, 360)]
    all_frames = []  # will be PIL images directly
    for i in range(N_SYNTH):
        w, h = random.choice(SYNTH_SIZES)
        arr = np.random.randint(50, 200, (h, w, 3), dtype=np.uint8)
        all_frames.append(Image.fromarray(arr))
    source = "synthetic"
    print(f"[SYNTHETIC DATA] Generated {N_SYNTH} synthetic frames for illustration.")

print(f"Analyzing {len(all_frames)} frames from {source} source...")

In [ ]:
widths, heights, aspect_ratios = [], [], []
mean_r, mean_g, mean_b = [], [], []

SAMPLE_SIZE = min(500, len(all_frames))
sample_indices = np.random.choice(len(all_frames), SAMPLE_SIZE, replace=False)

for idx in sample_indices:
    if HAS_FRAMES:
        img = _load_frame_pil(all_frames[idx])
    else:
        img = all_frames[idx]

    w, h = img.size
    widths.append(w)
    heights.append(h)
    aspect_ratios.append(w / h)

    arr = np.array(img, dtype=np.float32) / 255.0
    mean_r.append(arr[:, :, 0].mean())
    mean_g.append(arr[:, :, 1].mean())
    mean_b.append(arr[:, :, 2].mean())

widths  = np.array(widths)
heights = np.array(heights)
aspect_ratios = np.array(aspect_ratios)

print(f"Width  — mean: {widths.mean():.1f}  std: {widths.std():.1f}  min: {widths.min()}  max: {widths.max()}")
print(f"Height — mean: {heights.mean():.1f}  std: {heights.std():.1f}  min: {heights.min()}  max: {heights.max()}")
print(f"Aspect — mean: {aspect_ratios.mean():.3f}  std: {aspect_ratios.std():.3f}")
print()
print(f"Pixel mean  R: {np.mean(mean_r):.4f}  G: {np.mean(mean_g):.4f}  B: {np.mean(mean_b):.4f}")
print(f"Pixel std   R: {np.std(mean_r):.4f}   G: {np.std(mean_g):.4f}   B: {np.std(mean_b):.4f}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Width distribution
axes[0, 0].hist(widths,  bins=30, color="steelblue",   edgecolor="white")
axes[0, 0].set_title("Frame Width Distribution")
axes[0, 0].set_xlabel("Width (px)")
axes[0, 0].set_ylabel("Count")
axes[0, 0].axvline(widths.mean(), color="red", linestyle="--", label=f"mean={widths.mean():.0f}")
axes[0, 0].legend()

# Height distribution
axes[0, 1].hist(heights, bins=30, color="coral",       edgecolor="white")
axes[0, 1].set_title("Frame Height Distribution")
axes[0, 1].set_xlabel("Height (px)")
axes[0, 1].axvline(heights.mean(), color="red", linestyle="--", label=f"mean={heights.mean():.0f}")
axes[0, 1].legend()

# Aspect ratio
axes[0, 2].hist(aspect_ratios, bins=30, color="mediumseagreen", edgecolor="white")
axes[0, 2].set_title("Aspect Ratio (W/H)")
axes[0, 2].set_xlabel("Aspect ratio")
axes[0, 2].axvline(aspect_ratios.mean(), color="red", linestyle="--", label=f"mean={aspect_ratios.mean():.2f}")
axes[0, 2].legend()

# Per-channel pixel mean distribution
for ch_idx, (ch_means, color, name) in enumerate([(mean_r, "red", "R"), (mean_g, "green", "G"), (mean_b, "blue", "B")]):
    axes[1, ch_idx].hist(ch_means, bins=30, color=color, alpha=0.7, edgecolor="white")
    axes[1, ch_idx].set_title(f"Per-frame mean pixel value ({name} channel)")
    axes[1, ch_idx].set_xlabel("Mean pixel value (0–1)")
    axes[1, ch_idx].axvline(np.mean(ch_means), color="black", linestyle="--",
                            label=f"mean={np.mean(ch_means):.3f}")
    axes[1, ch_idx].legend()

plt.suptitle("RGB Frame Statistics", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("frame_statistics.png", bbox_inches="tight")
plt.show()

In [ ]:
# Scatter plot: width vs height (resolution cloud)
fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(widths, heights, c=aspect_ratios, cmap="viridis", alpha=0.6, s=20)
plt.colorbar(scatter, ax=ax, label="Aspect ratio (W/H)")
ax.set_title("Frame Resolution Scatter (width × height)")
ax.set_xlabel("Width (px)")
ax.set_ylabel("Height (px)")
plt.tight_layout()
plt.savefig("resolution_scatter.png", bbox_inches="tight")
plt.show()

### 3.1 Ejemplo de imagen

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 8))

example_indices = np.random.choice(len(all_frames), 8, replace=False)

for ax, idx in zip(axes.ravel(), example_indices):
    if HAS_FRAMES:
        img = _load_frame_pil(all_frames[idx])
        title = all_frames[idx].parent.name
    else:
        img = all_frames[idx]
        title = f"Synthetic frame {idx}"

    ax.imshow(img)
    ax.set_title(title, fontsize=9)
    ax.axis("off")

plt.suptitle("Sample RGB Frames", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("sample_frames.png", bbox_inches="tight")
plt.show()

---
## 4. Estadísticas de vídeos

Duración, resolución y FPS de los vídeos originales.

In [ ]:
import subprocess, json

def _probe_video(path: Path) -> dict:
    """Use ffprobe to extract video metadata."""
    cmd = [
        "ffprobe", "-v", "quiet",
        "-print_format", "json",
        "-show_streams", "-show_format",
        str(path),
    ]
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=30)
    return json.loads(result.stdout) if result.returncode == 0 else {}

if HAS_VIDEOS:
    video_files = sorted(VIDEOS_DIR.glob("**/*.MP4"))[:200]
    video_source = "real"
else:
    video_source = "synthetic"
    print("[SYNTHETIC DATA] Real videos not found. Generating synthetic statistics.")
    np.random.seed(42)
    N_VID = 300
    # Real EK100 videos: ~1-90 minutes, mostly 1080p @ 50 or 60 fps
    video_durations = np.random.exponential(scale=600, size=N_VID).clip(30, 5400)  # seconds
    video_fps_vals  = np.random.choice([50.0, 59.94, 60.0], N_VID, p=[0.5, 0.3, 0.2])
    video_widths    = np.random.choice([1920, 1280], N_VID, p=[0.85, 0.15])
    video_heights   = np.where(video_widths == 1920, 1080, 720)
    video_files = []
    print(f"Simulating {N_VID} videos")

print(f"Analyzing {len(video_files) if HAS_VIDEOS else N_VID} videos from {video_source} source...")

In [ ]:
if HAS_VIDEOS:
    durations, fps_list, vwidths, vheights = [], [], [], []
    for vf in video_files[:100]:  # limit probe for speed
        info = _probe_video(vf)
        for stream in info.get("streams", []):
            if stream.get("codec_type") == "video":
                dur = float(info.get("format", {}).get("duration", 0))
                durations.append(dur)
                # FPS: r_frame_rate is a fraction string like '60/1'
                r_fps = stream.get("r_frame_rate", "0/1")
                num, den = map(int, r_fps.split("/"))
                fps_list.append(num / den if den else 0)
                vwidths.append(int(stream.get("width", 0)))
                vheights.append(int(stream.get("height", 0)))
                break
    video_durations = np.array(durations)
    video_fps_vals  = np.array(fps_list)
    video_widths    = np.array(vwidths)
    video_heights   = np.array(vheights)
else:
    pass  # already generated above

if len(video_durations) > 0:
    print(f"Duration (s) — mean: {video_durations.mean():.1f}  median: {np.median(video_durations):.1f}  "
          f"min: {video_durations.min():.1f}  max: {video_durations.max():.1f}")
    print(f"FPS          — mean: {video_fps_vals.mean():.2f}  "
          f"unique: {np.unique(np.round(video_fps_vals, 2))}")
    print(f"Resolution   — most common: {video_widths[0]}×{video_heights[0]}")
    print(f"Avg duration : {video_durations.mean() / 60:.2f} min  ({video_durations.mean():.1f} s)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Duration histogram
axes[0].hist(video_durations / 60, bins=30, color="mediumpurple", edgecolor="white")
axes[0].axvline(video_durations.mean() / 60, color="red", linestyle="--",
                label=f"mean={video_durations.mean()/60:.1f} min")
axes[0].set_title("Video Duration Distribution")
axes[0].set_xlabel("Duration (minutes)")
axes[0].set_ylabel("Count")
axes[0].legend()

# FPS distribution
unique_fps, fps_counts = np.unique(np.round(video_fps_vals, 1), return_counts=True)
axes[1].bar([str(f) for f in unique_fps], fps_counts, color="darkorange", edgecolor="white")
axes[1].set_title("Frame Rate (FPS) Distribution")
axes[1].set_xlabel("FPS")
axes[1].set_ylabel("Count")

# Resolution
resolutions = [f"{w}×{h}" for w, h in zip(video_widths, video_heights)]
from collections import Counter
res_counts = Counter(resolutions).most_common(8)
axes[2].bar([r[0] for r in res_counts], [r[1] for r in res_counts], color="teal", edgecolor="white")
axes[2].set_title("Video Resolution Distribution")
axes[2].set_xlabel("Resolution")
axes[2].set_ylabel("Count")
axes[2].tick_params(axis="x", rotation=45)

plt.suptitle("Video Statistics", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("video_statistics.png", bbox_inches="tight")
plt.show()

---
## 5. Resumen ejecutivo

| Característica | Valor |
|----------------|-------|
| Splits disponibles | Train / Test |
| Segmentos de entrenamiento | ~3,760 |
| Segmentos de prueba | ~2,000 |
| Participantes | 45 |
| Entornos de cocina | 45 |
| Duración total | ~100 horas |
| FPS típico | 50–60 fps |
| Resolución típica | 1920×1080 |
| Fotogramas preprocesados | 256px lado corto, JPEG |
| Formato vídeo | .MP4 H.264 |
| Anotaciones | CSV + Pickle |
| Métricas de evaluación | mAP, nDCG |

### Referencias
- [EPIC-KITCHENS-100 Paper](https://arxiv.org/abs/2106.00182)
- [Dataset Website](https://epic-kitchens.github.io/2025)
- [Download Scripts](https://github.com/epic-kitchens/epic-kitchens-download-scripts)
- [Annotations Repo](https://github.com/epic-kitchens/epic-kitchens-100-annotations)
- [C5 Challenge](https://github.com/epic-kitchens/C5-Multi-Instance-Retrieval)